# RNN기반 분류기

In [1]:
# 데이터 로딩
from sklearn.datasets import fetch_20newsgroups

categories = ['comp.graphics', 'sci.space', 'rec.sport.baseball']
newsgroups = fetch_20newsgroups(subset='all', categories=categories)
X = newsgroups.data
y = newsgroups.target
print(newsgroups.target_names)
print(X[0])
print(y[0])


['comp.graphics', 'rec.sport.baseball', 'sci.space']
From: kjenks@gothamcity.jsc.nasa.gov
Subject: Life on Mars???
Organization: NASA/JSC/GM2, Space Shuttle Program Office 
X-Newsreader: TIN [version 1.1 PL8]
Lines: 12

I know it's only wishful thinking, with our current President,
but this is from last fall:

     "Is there life on Mars?  Maybe not now.  But there will be."
        -- Daniel S. Goldin, NASA Administrator, 24 August 1992

-- Ken Jenks, NASA/JSC/GM2, Space Shuttle Program Office
      kjenks@gothamcity.jsc.nasa.gov  (713) 483-4368

     "The man who makes no mistakes does not usually make
      anything."
        -- Edward John Phelps, American Diplomat/Lawyer (1825-1895)

2


In [2]:
# 데이터전처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X)
X_encoded = tokenizer.texts_to_sequences(X)
X_padded = pad_sequences(X_encoded, maxlen=max_len)
print(X_padded.shape)

(2954, 200)


In [3]:
# 데이터분리/텐서변환
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

X_train, X_test, y_train, y_test = \
    train_test_split(torch.tensor(X_padded), torch.tensor(y), test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = \
    train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# dataset/dataloader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 모델 생성
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        # embedding - lstm - dense
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, c) = self.lstm(x)
        out = self.fc(h[-1])
        return out

In [5]:
# 모델 학습
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # cuda 또는 cpu
embedding_dim = 100
hidden_size = 128

model = LSTMClassifier(vocab_size, embedding_dim, hidden_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습루프
train_losses, train_accs = [], []
val_losses, val_accs = [], []

epochs = 50
for epoch in range(epochs):

    # 학습
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.detach().cpu().item()
        pred = output.argmax(dim=1)
        train_correct += (pred == y_batch).sum().detach().cpu().item()
        train_total += len(y_batch)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # 검증
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)

            val_loss += loss.detach().cpu().item()
            pred = output.argmax(dim=1)
            val_correct += (pred == y_batch).sum().detach().cpu().item()
            val_total += len(y_batch)

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

    # 출력(train_loss, val_loss)
    print(f'Epoch {epoch + 1}/{epochs}: '
          f'Train Loss {train_loss:.4f}, '
          f'Train Acc {train_acc:.4f}, '
          f'Val Loss {val_loss:.4f}, '
          f'Val Acc {val_acc:.4f}, ')


Epoch 1/50: Train Loss 1.0651, Train Acc 0.4450, Val Loss 1.0127, Val Acc 0.5222, 
Epoch 2/50: Train Loss 0.9285, Train Acc 0.5974, Val Loss 0.8967, Val Acc 0.5941, 
Epoch 3/50: Train Loss 0.7536, Train Acc 0.6958, Val Loss 0.8269, Val Acc 0.6195, 
Epoch 4/50: Train Loss 0.5608, Train Acc 0.7942, Val Loss 0.7045, Val Acc 0.7167, 
Epoch 5/50: Train Loss 0.4188, Train Acc 0.8466, Val Loss 0.7040, Val Acc 0.7167, 
Epoch 6/50: Train Loss 0.2968, Train Acc 0.8937, Val Loss 0.6113, Val Acc 0.7548, 
Epoch 7/50: Train Loss 0.1979, Train Acc 0.9349, Val Loss 0.6103, Val Acc 0.7696, 
Epoch 8/50: Train Loss 0.1491, Train Acc 0.9540, Val Loss 0.5671, Val Acc 0.7907, 
Epoch 9/50: Train Loss 0.1034, Train Acc 0.9751, Val Loss 0.5194, Val Acc 0.8224, 
Epoch 10/50: Train Loss 0.0538, Train Acc 0.9862, Val Loss 0.5877, Val Acc 0.7992, 
Epoch 11/50: Train Loss 0.0591, Train Acc 0.9836, Val Loss 0.6014, Val Acc 0.7822, 
Epoch 12/50: Train Loss 0.0527, Train Acc 0.9899, Val Loss 0.6428, Val Acc 0.8118, 
E

In [6]:
# 모델 평가
# - 정답, 모델 예측값을 가지고, classification_report 작성
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        output = model(X_batch)
        loss = criterion(output, y_batch)
        pred = output.argmax(dim=1)

        all_preds.extend(pred.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=newsgroups.target_names))

                    precision    recall  f1-score   support

     comp.graphics       0.82      0.84      0.83       202
rec.sport.baseball       0.86      0.87      0.87       202
         sci.space       0.79      0.76      0.78       187

          accuracy                           0.83       591
         macro avg       0.82      0.82      0.82       591
      weighted avg       0.83      0.83      0.83       591



## 사전학습된 임베딩 적용하기

In [7]:
%pip install gensim -q

Note: you may need to restart the kernel to use updated packages.


In [10]:
from gensim.models import FastText

fasttext_model = FastText.load('ted_en_fasttext.model')
print(fasttext_model.vector_size)

100


In [11]:
import numpy as np

embedding_dim = fasttext_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

word_index = tokenizer.word_index # 38000
word_index = {word:index \
              for word, index in word_index.items() \
                if index < vocab_size}
print(len(word_index)) # 10000

for word, index in word_index.items():
    if word in fasttext_model.wv:
        embedding_matrix[index] = fasttext_model.wv[word]

9999


In [18]:
# 모델 생성
import torch.nn as nn

class LSTMClassifier2(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, hidden_size):
        super().__init__()
        # embedding - lstm - dense
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, c) = self.lstm(x)
        out = self.fc(h[-1])
        return out


In [19]:
# 모델 학습
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # cuda 또는 cpu
embedding_dim = 100
hidden_size = 128

model = LSTMClassifier2(vocab_size, embedding_dim, embedding_matrix, hidden_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 학습루프
train_losses, train_accs = [], []
val_losses, val_accs = [], []

epochs = 100
for epoch in range(epochs):

    # 학습
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.detach().cpu().item()
        pred = output.argmax(dim=1)
        train_correct += (pred == y_batch).sum().detach().cpu().item()
        train_total += len(y_batch)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # 검증
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)

            val_loss += loss.detach().cpu().item()
            pred = output.argmax(dim=1)
            val_correct += (pred == y_batch).sum().detach().cpu().item()
            val_total += len(y_batch)

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

    # 출력(train_loss, val_loss)
    print(f'Epoch {epoch + 1}/{epochs}: '
          f'Train Loss {train_loss:.4f}, '
          f'Train Acc {train_acc:.4f}, '
          f'Val Loss {val_loss:.4f}, '
          f'Val Acc {val_acc:.4f}, ')


Epoch 1/100: Train Loss 1.0983, Train Acc 0.3556, Val Loss 1.0947, Val Acc 0.3848, 
Epoch 2/100: Train Loss 1.0946, Train Acc 0.3931, Val Loss 1.0907, Val Acc 0.3679, 
Epoch 3/100: Train Loss 1.0913, Train Acc 0.3772, Val Loss 1.0868, Val Acc 0.3679, 
Epoch 4/100: Train Loss 1.0873, Train Acc 0.4624, Val Loss 1.0818, Val Acc 0.4609, 
Epoch 5/100: Train Loss 1.0816, Train Acc 0.5074, Val Loss 1.0751, Val Acc 0.4905, 
Epoch 6/100: Train Loss 1.0739, Train Acc 0.5217, Val Loss 1.0611, Val Acc 0.4989, 
Epoch 7/100: Train Loss 1.0539, Train Acc 0.5677, Val Loss 1.0227, Val Acc 0.5497, 
Epoch 8/100: Train Loss 0.9623, Train Acc 0.5958, Val Loss 0.8899, Val Acc 0.5983, 
Epoch 9/100: Train Loss 0.8311, Train Acc 0.6635, Val Loss 0.8239, Val Acc 0.6512, 
Epoch 10/100: Train Loss 0.8709, Train Acc 0.6106, Val Loss 0.8348, Val Acc 0.6490, 
Epoch 11/100: Train Loss 0.7960, Train Acc 0.6884, Val Loss 0.7942, Val Acc 0.7167, 
Epoch 12/100: Train Loss 0.7701, Train Acc 0.6968, Val Loss 1.1839, Val Ac

In [20]:
# 모델 평가
# - 정답, 모델 예측값을 가지고, classification_report 작성
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        output = model(X_batch)
        loss = criterion(output, y_batch)
        pred = output.argmax(dim=1)

        all_preds.extend(pred.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=newsgroups.target_names))

                    precision    recall  f1-score   support

     comp.graphics       0.87      0.95      0.91       202
rec.sport.baseball       0.96      0.93      0.94       202
         sci.space       0.95      0.89      0.92       187

          accuracy                           0.92       591
         macro avg       0.93      0.92      0.92       591
      weighted avg       0.93      0.92      0.92       591

